# Error Analysis: From Failure Modes to Prompt Improvement
This notebook demonstrates how to systematically evaluate and improve an LLM agent using qualitative error analysis rather than automated scores.
Below are steps in error analysis:

1. **Define** evaluation criteria
2. **Build** baseline agent
3. **Collect** expert feedback
4. **Extract** failure modes
5. **Analyze** failure frequency
6. **Improve** prompt based on evidence

In [1]:
!pip install groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 2.4 MB/s eta 0:00:00


In [2]:
import getpass
import os
import json
import re
from collections import Counter
from groq import Groq

os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ API key: ")
client = Groq()

Enter your GROQ API key: ··········


---
## Step 1: Define Evaluation Criteria

What makes a "good" product description?

In [3]:
evaluation_criteria = {
    "Factual Consistency": "Description must only include facts inferable from the product title",
    "No Hallucinations": "No made-up specifications, prices, or features not in the title",
    "Clarity": "Clear, readable language appropriate for e-commerce",
    "Appropriate Tone": "Professional, helpful, not overly salesy or hyperbolic",
    "Completeness": "Covers the key aspects implied by the product title"
}

print("Quality Criteria for Product Descriptions:")
print("=" * 50)
for criterion, definition in evaluation_criteria.items():
    print(f"• {criterion}: {definition}")

Quality Criteria for Product Descriptions:
• Factual Consistency: Description must only include facts inferable from the product title
• No Hallucinations: No made-up specifications, prices, or features not in the title
• Clarity: Clear, readable language appropriate for e-commerce
• Appropriate Tone: Professional, helpful, not overly salesy or hyperbolic
• Completeness: Covers the key aspects implied by the product title


---
## Step 2: Build Agent V1 (Baseline Prompt)

A simple prompt-only agent that generates product descriptions from titles.

In [4]:
PROMPT_V1 = """Generate a product description for an e-commerce listing.

Product Title: {title}

Write a short description (2-3 sentences)."""

def agent_v1(title: str) -> str:
    """Baseline agent with minimal prompt"""
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": PROMPT_V1.format(title=title)}],
        temperature=0.7
    )
    return response.choices[0].message.content.strip()

# Test the agent
test_title = "Sony WH-1000XM5 Wireless Noise Cancelling Headphones - Black"
print(f"Title: {test_title}")
print(f"\nAgent V1 Output:\n{agent_v1(test_title)}")

Title: Sony WH-1000XM5 Wireless Noise Cancelling Headphones - Black

Agent V1 Output:
"Experience unparalleled audio quality with the Sony WH-1000XM5 Wireless Noise Cancelling Headphones. Equipped with industry-leading noise cancellation technology, these sleek black headphones provide immersive sound and a distraction-free listening experience. With advanced features like quick attention mode and smart listening, you can focus on what matters most."


---
## Step 3: Create 10 Synthetic Test Inputs

Diverse product titles across categories.

In [5]:
test_products_v1 = [
    # Electronics
    "Apple MacBook Air M2 13-inch Laptop - Midnight",
    "Samsung 65\" 4K QLED Smart TV",
    "Anker USB-C 10000mAh Portable Charger",
    # Apparel
    "Nike Air Max 270 Men's Running Shoes - White/Black",
    "Levi's 501 Original Fit Jeans - Dark Wash",
    # Home & Kitchen
    "Instant Pot Duo 7-in-1 Electric Pressure Cooker 6 Quart",
    "Dyson V15 Detect Cordless Vacuum Cleaner",
    # Beauty
    "The Ordinary Niacinamide 10% + Zinc 1% Serum",
    # Sports
    "Yeti Rambler 26oz Vacuum Insulated Water Bottle - Navy",
    # Books/Media
    "Kindle Paperwhite 16GB - 6.8\" Display"
]

print(f"Created {len(test_products_v1)} test products")
for i, title in enumerate(test_products_v1, 1):
    print(f"  {i}. {title}")

Created 10 test products
  1. Apple MacBook Air M2 13-inch Laptop - Midnight
  2. Samsung 65" 4K QLED Smart TV
  3. Anker USB-C 10000mAh Portable Charger
  4. Nike Air Max 270 Men's Running Shoes - White/Black
  5. Levi's 501 Original Fit Jeans - Dark Wash
  6. Instant Pot Duo 7-in-1 Electric Pressure Cooker 6 Quart
  7. Dyson V15 Detect Cordless Vacuum Cleaner
  8. The Ordinary Niacinamide 10% + Zinc 1% Serum
  9. Yeti Rambler 26oz Vacuum Insulated Water Bottle - Navy
  10. Kindle Paperwhite 16GB - 6.8" Display


---
## Step 4: Collect Expert Feedback

Run Agent V1 on all products and simulate expert feedback.

In a real scenario, human experts review outputs. Here we simulate realistic feedback.

In [6]:
# Generate outputs for all test products
v1_outputs = []
print("Generating Agent V1 outputs...")
for i, title in enumerate(test_products_v1):
    output = agent_v1(title)
    v1_outputs.append({"title": title, "output": output})
    print(f"  [{i+1}/{len(test_products_v1)}] Done")

print("\nSample outputs:")
for item in v1_outputs[:3]:
    print(f"\nTitle: {item['title']}")
    print(f"Output: {item['output'][:150]}...")

Generating Agent V1 outputs...
  [1/10] Done
  [2/10] Done
  [3/10] Done
  [4/10] Done
  [5/10] Done
  [6/10] Done
  [7/10] Done
  [8/10] Done
  [9/10] Done
  [10/10] Done

Sample outputs:

Title: Apple MacBook Air M2 13-inch Laptop - Midnight
Output: Introducing the Apple MacBook Air M2 13-inch Laptop in Midnight, a sleek and powerful device that combines stunning design with exceptional performanc...

Title: Samsung 65" 4K QLED Smart TV
Output: Experience breathtaking visuals on the Samsung 65" 4K QLED Smart TV, featuring stunning color accuracy and an impressive 4K resolution that brings you...

Title: Anker USB-C 10000mAh Portable Charger
Output: "Stay powered on-the-go with our Anker USB-C 10000mAh Portable Charger. This compact and lightweight charger provides up to 3 full charges for most sm...


In [7]:
# Simulate expert feedback using LLM (in production, humans would provide this)
def simulate_expert_feedback(title: str, output: str) -> str:
    """Simulate expert critique of the output"""
    prompt = f"""You are an e-commerce quality reviewer. Critique this product description.

Product Title: {title}
Generated Description: {output}

Identify specific problems with this description. Be critical and specific.
Focus on: factual errors, hallucinations, unclear language, wrong tone, missing info.

Write 1-2 sentences describing what's wrong (or "No issues" if acceptable)."""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )
    return response.choices[0].message.content.strip()

# Collect feedback for all outputs
expert_feedback = []
print("Collecting expert feedback...")
for i, item in enumerate(v1_outputs):
    feedback = simulate_expert_feedback(item["title"], item["output"])
    expert_feedback.append({
        "title": item["title"],
        "output": item["output"],
        "feedback": feedback
    })
    print(f"  [{i+1}/{len(v1_outputs)}] Done")

print("\nExpert Feedback Summary:")
print("=" * 60)
for item in expert_feedback:
    print(f"\nTitle: {item['title'][:50]}...")
    print(f"Feedback: {item['feedback']}")

  [1/10] Done
  [2/10] Done
  [3/10] Done
  [4/10] Done
  [5/10] Done
  [6/10] Done
  [7/10] Done
  [8/10] Done
  [9/10] Done
  [10/10] Done

Expert Feedback Summary:

Title: Apple MacBook Air M2 13-inch Laptop - Midnight...
Feedback: The description lacks specific details about the laptop's technical specifications, such as storage capacity, RAM, and graphics capabilities, which are crucial for informed purchasing decisions. Additionally, the phrase "all-day battery life" is vague and should be replaced with a more precise estimate of battery life in hours to provide a clearer understanding of the laptop's capabilities.

Title: Samsung 65" 4K QLED Smart TV...
Feedback: The description lacks specific details about the TV's features, such as its HDR support, refresh rate, and smart TV platform, which are crucial for informed purchasing decisions. Additionally, the phrase "elevate your home's audiovisual experience" is somewhat vague and doesn't provide concrete information about the TV'

---
## Step 5: Identify Failure Modes Using LLM

Analyze outputs and feedback to extract recurring failure patterns.

In [8]:
# Prepare all feedback for analysis
all_feedback = "\n\n".join([
    f"Product: {item['title']}\nOutput: {item['output']}\nFeedback: {item['feedback']}"
    for item in expert_feedback
])

extraction_prompt = f"""Analyze these product description outputs and expert feedback.
Extract the recurring failure modes (error patterns).

{all_feedback}

List 5-7 distinct failure modes you observe. For each:
- Give it a short name (2-4 words)
- Briefly describe what the error is
- Give one example from above

Format as:
1. [Name]: [Description]. Example: [quote]
2. ..."""

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": extraction_prompt}],
    temperature=0.2
)

extracted_failures = response.choices[0].message.content
print("Extracted Failure Modes:")
print("=" * 60)
print(extracted_failures)

Extracted Failure Modes:
Here are 7 distinct failure modes observed:

1. **Lack of Specifics**: The error is the absence of crucial technical details or specifications that are necessary for informed purchasing decisions. Example: "The description lacks specific details about the laptop's technical specifications, such as storage capacity, RAM, and graphics capabilities..."
2. **Vagueness**: The error is the use of vague or ambiguous language that does not provide clear information about the product's features or capabilities. Example: "The phrase 'all-day battery life' is vague and should be replaced with a more precise estimate of battery life in hours..."
3. **Hallucinations**: The error is the inclusion of false or misleading information that is not supported by facts. Example: "The claim of providing 'up to 3 full charges for most smartphones' is vague and may not be accurate for all smartphone models..."
4. **Overly Promotional**: The error is the use of overly promotional langua

---
## Step 6: Create Failure Mode Taxonomy

Consolidate into a structured taxonomy for tagging. The goal is not to perfectly classify errors, but to create a stable taxonomy that can be reused across datasets.

In [9]:
# Define failure mode taxonomy based on extraction
failure_taxonomy = {
    "HALLUCINATION": "Made-up specs, features, or claims not in the title (e.g., inventing battery life, prices)",
    "OVERLY_PROMOTIONAL": "Excessive marketing language, superlatives, or hype (e.g., 'revolutionary', 'best ever')",
    "VAGUE_GENERIC": "Generic filler that could apply to any product (e.g., 'high quality', 'great value')",
    "MISSING_KEY_INFO": "Fails to mention important details from the title (e.g., ignoring color, size)",
    "WRONG_CATEGORY": "Misunderstands the product type or use case",
    "FORMATTING_ISSUES": "Poor structure, awkward sentences, or inconsistent style",
    "FACTUAL_ERROR": "Incorrect factual claims that contradict the title or common knowledge"
}

print("Failure Mode Taxonomy:")
print("=" * 60)
for code, description in failure_taxonomy.items():
    print(f"\n{code}:")
    print(f"  {description}")

Failure Mode Taxonomy:

HALLUCINATION:
  Made-up specs, features, or claims not in the title (e.g., inventing battery life, prices)

OVERLY_PROMOTIONAL:
  Excessive marketing language, superlatives, or hype (e.g., 'revolutionary', 'best ever')

VAGUE_GENERIC:
  Generic filler that could apply to any product (e.g., 'high quality', 'great value')

MISSING_KEY_INFO:
  Fails to mention important details from the title (e.g., ignoring color, size)

WRONG_CATEGORY:
  Misunderstands the product type or use case

FORMATTING_ISSUES:
  Poor structure, awkward sentences, or inconsistent style

FACTUAL_ERROR:
  Incorrect factual claims that contradict the title or common knowledge


---
## Step 7: Tag Errors on 20 New Test Cases

Test whether failure modes generalize to new products.

In [10]:
# 20 new test products
test_products_v2 = [
    "Bose QuietComfort Ultra Earbuds - Black",
    "Canon EOS R6 Mark II Mirrorless Camera Body",
    "LG 27\" UltraGear QHD Gaming Monitor 165Hz",
    "Patagonia Men's Better Sweater Fleece Jacket - Grey",
    "Allbirds Women's Tree Runners - Natural White",
    "KitchenAid Artisan 5-Quart Stand Mixer - Empire Red",
    "Roomba j7+ Self-Emptying Robot Vacuum",
    "Olaplex No. 3 Hair Perfector Treatment",
    "CeraVe Hydrating Facial Cleanser 16 oz",
    "Hydro Flask 32oz Wide Mouth - Pacific",
    "Theragun Prime Percussion Massage Device",
    "Nintendo Switch OLED Model - White",
    "Herman Miller Aeron Chair - Size B Graphite",
    "Sonos Arc Premium Smart Soundbar - Black",
    "Le Creuset Signature 5.5 Qt Dutch Oven - Flame",
    "Osprey Atmos AG 65 Backpack - Men's Large",
    "Breville Barista Express Espresso Machine",
    "Ray-Ban Aviator Classic Sunglasses - Gold/Green",
    "Vitamix E310 Explorian Blender 48oz",
    "Apple Watch Ultra 2 49mm - Titanium"
]

print(f"Created {len(test_products_v2)} new test products")

Created 20 new test products


In [11]:
def tag_failures(title: str, output: str, taxonomy: dict) -> list:
    """Tag output with failure modes from taxonomy"""
    taxonomy_text = "\n".join([f"- {k}: {v}" for k, v in taxonomy.items()])

    prompt = f"""Analyze this product description and tag it with failure modes.

Product Title: {title}
Generated Description: {output}

Failure Mode Taxonomy:
{taxonomy_text}

Which failure modes apply to this output? List ONLY the codes that apply.
If the output is good, respond with: NONE

Format: CODE1, CODE2, ... (or NONE)"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    result = response.choices[0].message.content.strip().upper()
    if "NONE" in result:
        return []
    # Extract valid codes
    codes = [c.strip() for c in result.split(",")]
    return [c for c in codes if c in taxonomy]

# Generate outputs and tag failures
tagged_results = []
print("Generating outputs and tagging failures...")
for i, title in enumerate(test_products_v2):
    output = agent_v1(title)
    tags = tag_failures(title, output, failure_taxonomy)
    tagged_results.append({"title": title, "output": output, "tags": tags})
    print(f"  [{i+1}/{len(test_products_v2)}] Tags: {tags if tags else 'NONE'}")

print("\nTagging complete!")

Generating outputs and tagging failures...
  [1/20] Tags: ['OVERLY_PROMOTIONAL', 'VAGUE_GENERIC']
  [2/20] Tags: ['OVERLY_PROMOTIONAL', 'VAGUE_GENERIC']
  [3/20] Tags: ['OVERLY_PROMOTIONAL', 'VAGUE_GENERIC']
  [4/20] Tags: ['VAGUE_GENERIC', 'OVERLY_PROMOTIONAL']
  [5/20] Tags: ['OVERLY_PROMOTIONAL', 'VAGUE_GENERIC']
  [6/20] Tags: ['OVERLY_PROMOTIONAL', 'VAGUE_GENERIC']
  [7/20] Tags: ['OVERLY_PROMOTIONAL', 'VAGUE_GENERIC']
  [8/20] Tags: ['OVERLY_PROMOTIONAL', 'VAGUE_GENERIC']
  [9/20] Tags: ['OVERLY_PROMOTIONAL', 'VAGUE_GENERIC']
  [10/20] Tags: ['VAGUE_GENERIC', 'OVERLY_PROMOTIONAL']
  [11/20] Tags: ['OVERLY_PROMOTIONAL', 'VAGUE_GENERIC']
  [12/20] Tags: ['OVERLY_PROMOTIONAL', 'VAGUE_GENERIC']
  [13/20] Tags: ['OVERLY_PROMOTIONAL', 'VAGUE_GENERIC']
  [14/20] Tags: ['OVERLY_PROMOTIONAL', 'VAGUE_GENERIC']
  [15/20] Tags: ['OVERLY_PROMOTIONAL', 'VAGUE_GENERIC']
  [16/20] Tags: ['OVERLY_PROMOTIONAL', 'VAGUE_GENERIC']
  [17/20] Tags: ['OVERLY_PROMOTIONAL', 'VAGUE_GENERIC']
  [18/20] Tags

---
## Step 8: Analyze Failure Mode Frequency

Count occurrences to identify the most impactful issues.

In [12]:
# Count failure modes
all_tags = [tag for result in tagged_results for tag in result["tags"]]
tag_counts = Counter(all_tags)

# Calculate statistics
total_outputs = len(tagged_results)
outputs_with_issues = sum(1 for r in tagged_results if r["tags"])

print("Failure Mode Analysis")
print("=" * 60)
print(f"Total outputs analyzed: {total_outputs}")
print(f"Outputs with issues: {outputs_with_issues} ({outputs_with_issues/total_outputs*100:.0f}%)")
print(f"Clean outputs: {total_outputs - outputs_with_issues}")
print()
print("Failure Mode Frequency:")
print("-" * 40)

for code in failure_taxonomy:
    count = tag_counts.get(code, 0)
    pct = count / total_outputs * 100
    bar = "█" * int(pct / 5)
    print(f"{code:<20} {count:>3} ({pct:>5.1f}%) {bar}")

# Identify top issues
top_issues = tag_counts.most_common(3)
print("\n" + "=" * 60)
print("TOP 3 ISSUES TO FIX:")
for i, (code, count) in enumerate(top_issues, 1):
    print(f"  {i}. {code} ({count} occurrences)")
    print(f"     → {failure_taxonomy[code]}")

Failure Mode Analysis
Total outputs analyzed: 20
Outputs with issues: 20 (100%)
Clean outputs: 0

Failure Mode Frequency:
----------------------------------------
HALLUCINATION          0 (  0.0%) 
OVERLY_PROMOTIONAL    20 (100.0%) ████████████████████
VAGUE_GENERIC         20 (100.0%) ████████████████████
MISSING_KEY_INFO       0 (  0.0%) 
WRONG_CATEGORY         0 (  0.0%) 
FORMATTING_ISSUES      0 (  0.0%) 
FACTUAL_ERROR          0 (  0.0%) 

TOP 3 ISSUES TO FIX:
  1. OVERLY_PROMOTIONAL (20 occurrences)
     → Excessive marketing language, superlatives, or hype (e.g., 'revolutionary', 'best ever')
  2. VAGUE_GENERIC (20 occurrences)
     → Generic filler that could apply to any product (e.g., 'high quality', 'great value')


---
## Step 9: Improve Prompt Based on Evidence

Create Prompt V2 that explicitly addresses the top failure modes.

In [13]:
# Build improved prompt targeting top failure modes
top_failure_codes = [code for code, _ in top_issues]

# Map failure codes to specific instructions
fix_instructions = {
    "HALLUCINATION": "Only include information explicitly stated in the product title. Do not invent specs, prices, or features.",
    "OVERLY_PROMOTIONAL": "Use neutral, informative language. Avoid superlatives like 'best', 'revolutionary', or 'amazing'.",
    "VAGUE_GENERIC": "Be specific. Avoid generic phrases like 'high quality' or 'great value' that could apply to any product.",
    "MISSING_KEY_INFO": "Include all key details from the title: brand, model, size, color, and distinguishing features.",
    "WRONG_CATEGORY": "Correctly identify the product category and describe appropriate use cases.",
    "FORMATTING_ISSUES": "Write clear, concise sentences. Use consistent formatting.",
    "FACTUAL_ERROR": "Only state facts that can be directly inferred from the product title."
}

# Build targeted instructions
targeted_rules = "\n".join([f"- {fix_instructions[code]}" for code in top_failure_codes])

PROMPT_V2 = f"""Generate a product description for an e-commerce listing.

Product Title: {{title}}

RULES:
{targeted_rules}

Write a short description (2-3 sentences) that accurately represents the product."""

print("PROMPT V2 (Evidence-Based Improvement)")
print("=" * 60)
print(PROMPT_V2)

def agent_v2(title: str) -> str:
    """Improved agent with evidence-based prompt"""
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": PROMPT_V2.format(title=title)}],
        temperature=0.7
    )
    return response.choices[0].message.content.strip()

PROMPT V2 (Evidence-Based Improvement)
Generate a product description for an e-commerce listing.

Product Title: {title}

RULES:
- Use neutral, informative language. Avoid superlatives like 'best', 'revolutionary', or 'amazing'.
- Be specific. Avoid generic phrases like 'high quality' or 'great value' that could apply to any product.

Write a short description (2-3 sentences) that accurately represents the product.


In [14]:
# Compare V1 vs V2 on same products
print("Comparing Agent V1 vs V2")
print("=" * 70)

comparison_products = test_products_v2[:5]  # Compare on first 5

for title in comparison_products:
    v1_out = agent_v1(title)
    v2_out = agent_v2(title)
    v1_tags = tag_failures(title, v1_out, failure_taxonomy)
    v2_tags = tag_failures(title, v2_out, failure_taxonomy)

    print(f"\n{'─'*70}")
    print(f"Product: {title}")
    print(f"\nV1: {v1_out}")
    print(f"V1 Issues: {v1_tags if v1_tags else 'None'}")
    print(f"\nV2: {v2_out}")
    print(f"V2 Issues: {v2_tags if v2_tags else 'None'}")

Comparing Agent V1 vs V2

──────────────────────────────────────────────────────────────────────
Product: Bose QuietComfort Ultra Earbuds - Black

V1: "Immerse yourself in unparalleled sound with the Bose QuietComfort Ultra Earbuds. These sleek black earbuds deliver industry-leading noise cancellation, allowing you to focus on your favorite music or podcasts without distraction. With up to 24 hours of battery life, you can enjoy uninterrupted listening all day long."
V1 Issues: ['OVERLY_PROMOTIONAL', 'VAGUE_GENERIC', 'HALLUCINATION']

V2: The Bose QuietComfort Ultra Earbuds - Black are designed to provide active noise cancellation and comfortable listening. They feature 11mm dynamic drivers and a proprietary noise-rejection system to minimize external distractions, while a long-lasting battery offers up to 32 hours of playback. Compatible with both iOS and Android devices, these earbuds support wireless charging and voice assistant integration.
V2 Issues: None

────────────────────────

In [15]:
# Full comparison on all 20 products
print("Running full comparison on 20 products...")

v1_all_tags = []
v2_all_tags = []

for i, title in enumerate(test_products_v2):
    # V1
    v1_out = tagged_results[i]["output"]  # Reuse from earlier
    v1_tags = tagged_results[i]["tags"]
    v1_all_tags.extend(v1_tags)

    # V2
    v2_out = agent_v2(title)
    v2_tags = tag_failures(title, v2_out, failure_taxonomy)
    v2_all_tags.extend(v2_tags)

    print(f"  [{i+1}/20] V1: {len(v1_tags)} issues, V2: {len(v2_tags)} issues")

# Summary
print("\n" + "=" * 60)
print("IMPROVEMENT SUMMARY")
print("=" * 60)
print(f"{'Metric':<30} {'V1':>10} {'V2':>10} {'Change':>12}")
print("-" * 60)

v1_total = len(v1_all_tags)
v2_total = len(v2_all_tags)
change = ((v2_total - v1_total) / v1_total * 100) if v1_total > 0 else 0

print(f"{'Total Issues':<30} {v1_total:>10} {v2_total:>10} {change:>+11.1f}%")

# Per failure mode comparison
v1_counts = Counter(v1_all_tags)
v2_counts = Counter(v2_all_tags)

for code in top_failure_codes:
    v1_c = v1_counts.get(code, 0)
    v2_c = v2_counts.get(code, 0)
    c = ((v2_c - v1_c) / v1_c * 100) if v1_c > 0 else 0
    print(f"{code:<30} {v1_c:>10} {v2_c:>10} {c:>+11.1f}%")

Running full comparison on 20 products...
  [1/20] V1: 2 issues, V2: 0 issues
  [2/20] V1: 2 issues, V2: 0 issues
  [3/20] V1: 2 issues, V2: 0 issues
  [4/20] V1: 2 issues, V2: 0 issues
  [5/20] V1: 2 issues, V2: 0 issues
  [6/20] V1: 2 issues, V2: 0 issues
  [7/20] V1: 2 issues, V2: 0 issues
  [8/20] V1: 2 issues, V2: 2 issues
  [9/20] V1: 2 issues, V2: 0 issues
  [10/20] V1: 2 issues, V2: 2 issues
  [11/20] V1: 2 issues, V2: 0 issues
  [12/20] V1: 2 issues, V2: 0 issues
  [13/20] V1: 2 issues, V2: 0 issues
  [14/20] V1: 2 issues, V2: 2 issues
  [15/20] V1: 2 issues, V2: 0 issues
  [16/20] V1: 2 issues, V2: 0 issues
  [17/20] V1: 2 issues, V2: 0 issues
  [18/20] V1: 2 issues, V2: 1 issues
  [19/20] V1: 2 issues, V2: 0 issues
  [20/20] V1: 2 issues, V2: 0 issues

IMPROVEMENT SUMMARY
Metric                                 V1         V2       Change
------------------------------------------------------------
Total Issues                           40          7       -82.5%
OVERLY_PROMOT

---
## Summary

### The Error Analysis Process

```
1. Define Criteria    →  What does "good" look like?
2. Build Baseline     →  Simple V1 agent
3. Collect Feedback   →  Expert critiques on outputs
4. Extract Patterns   →  LLM identifies recurring issues
5. Create Taxonomy    →  5-7 named failure modes
6. Tag & Analyze      →  Measure frequency on new data
7. Improve Prompt     →  Target top failure modes
```

### Key Insights

| Principle | Why It Matters |
|-----------|----------------|
| **Evidence-based** | Improvements target observed issues, not intuition |
| **Taxonomy first** | Named categories enable systematic tracking |
| **Prioritize by frequency** | Fix high-impact issues first |
| **Measure before/after** | Verify improvements quantitatively |

### When to Use This Approach

- When prompt tuning isn't improving quality
- When you need to explain *why* outputs are bad
- When building quality guardrails
- When prioritizing engineering effort